In [1]:
#imports 

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import math
import copy
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace
from pathlib import Path

In [2]:
class InputEmbeddings(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        
    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)

In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, len_seq, d_model, dropout):
        super().__init__()
        
        self.len_seq = len_seq
        self.d_model = d_model
        self.dropout = nn.Dropout(dropout)
        
        pe = torch.zeros(len_seq, d_model)
        pos = torch.arange(0, len_seq, dtype=torch.float).unsqueeze(1)
        
        exp_term = torch.arange(0, d_model, step=2) / d_model
        
        pe[:, 0::2] = torch.sin(pos / 1e4 ** exp_term)
        
        pe[:, 1::2] = torch.cos(pos / 1e4 ** exp_term)
        
        pe = pe.unsqueeze(0)
        
        self.register_buffer('pos_enc', pe)
        
    def forward(self, x):
        x = x + (self.pe[:, :x.shape[1], :]).requires_grad(False)
        return self.dropout(x)

In [4]:
class MultiHeadAttentionBlock(nn.Module):
    def __init__(self, d_model, h, dropout):
        super().__init__()
        
        self.d_model = d_model
        self.h = h
        self.dropout = nn.Dropout(dropout)
        
        if d_model % h  != 0:
            raise ValueError('d_model is not divisible by h')
        
        self.d_k = d_model // h
        
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)
        
    def SelfAttention(self, query, key, val, mask, dropout):
        
        attention_scores = (query @ key.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            attention_scores.masked_fill_(mask == 0, -1e9)
        
        attention_scores = attention_scores.softmax(dim=-1)
        
        if dropout is not None:
            attention_scores  = dropout(attention_scores)
        
        
        return (attention_scores @ val), attention_scores
        
        
    def forward(self, q, k, v, mask):

        query = self.w_q(q)
        key = self.w_k(k)
        val = self.w_v(v)
        
        query = query.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1, 2)
        key = key.view(key.shape[0], key.shape[1], self.h, self.d_k).transpose(1, 2)     
        val = val.view(val.shape[0], val.shape[1], self.h, self.d_k).transpose(1, 2)  
        
    
        x, self.attention_scores = self.attention(query, key, val, mask, self.dropout)
                
        x = x.transpose(1, 2).contiguous().view(x.shape[0], -1, self.h * self.d_k)
        
        return self.w_o(x)

In [5]:
class FeedForwardBLock(nn.Module):
    def __init__(self, d_model, dff, droput):
        super().__init__()
        
        self.l1 = nn.Linear(d_model, dff)
        self.l2 = nn.Linear(dff, d_model)
        self.dropout = nn.Dropout(droput)
        
    def forward(self, x):
            
        x = nn.ReLU(self.l1(x))
        x = self.dropout(x)
        x = self.l2(x)
        
        return x

In [6]:
class LayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        
        self.alpha = nn.Parameter(torch.ones(d_model))
        self.beta  = nn.Parameter(torch.zeros(d_model))
        self.epsilon = 1e-6
        
    def forward(self, x):
        
        mean = x.mean(dim= -1, keepdim=True)
        
        std = x.std(dim= -1, keepdim=True)
        
        
        return self.alpha * (x - mean) / (std + self.epsilon) + self.beta

In [7]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, multi_head_attention_block, feed_forward_block, dropout):
        super().__init__()
        self.multi_head_attention_block = multi_head_attention_block
        self.feed_forward_block = feed_forward_block
        
 
        self.norm1 = LayerNorm(d_model) 
        self.norm2 = LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        
        attention_out = self.multi_head_attention_block(x, x, x, src_mask)
        
        x = self.norm1(x + self.dropout(attention_out))
      
        ff_out = self.feed_forward_block(x)
        
        x = self.norm2(x + self.dropout(ff_out))

        return x

In [8]:
class Encoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
    
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(N)])
 
        self.norm = LayerNorm(layer.norm1.alpha.shape[0])

    def forward(self, x, mask):
 
        for layer in self.layers:
            x = layer(x, mask)
        
        return self.norm(x)

In [9]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, self_attention_block, cross_attention_block, feed_forward_net, dropout):
        super().__init__()
        
        self.self_attention_block = self_attention_block
        self.cross_attention_block = cross_attention_block
        self.feed_forward_net = feed_forward_net
        
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)
        self.norm3 = LayerNorm(d_model)
        
        self.dropout = nn.Dropout(dropout)
        
        
    def forward(self, x, enc_out, src_mask, target_mask):
        
        attention_out = self.self_attention_block(x, x, x, target_mask)
        x = self.norm1(x + self.dropout(attention_out))
        
        cross_attn_out = self.cross_attention_block(x, enc_out, enc_out, src_mask) 
        x = self.norm2(x + self.dropout(cross_attn_out))
        
        ff__out = self.feed_forward_net(x)
        x = self.norm3(x + self.dropout(ff__out))
        
        return x        

In [10]:
class Decoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(N)])
 
        self.norm = LayerNorm(layer.norm1.alpha.shape[0]) # too lazy to add d_model in the func param

    def forward(self, x, enc_out, src_mask, target_mask):
 
        for layer in self.layers:
            x = layer(x, enc_out, src_mask, target_mask)
        
        return self.norm(x)

In [11]:
class ProjectionLayer(nn.Module):
    def __init__(self, d_model, vocab_size):
        super().__init__()
        
        self.proj_layer = nn.Linear(d_model, vocab_size)
        
    def forward(self, x):
        return torch.softmax(self.proj_layer(x), dim=-1)

In [12]:
class Transformer(nn.Module):
    def __init__(self, encoder, decoder, src_embed, target_embed, src_pos, target_pos, projection_layer):
        super().__init__()
        
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.src_pos = src_pos
        self.target_embed = target_embed
        self.target_pos = target_pos
        self.projection_layer = projection_layer
        
    def encode(self, src, src_mask):
        src = self.src_embed(src)
        src = self.src_pos(src)
        src_out = self.encoder(src, src_mask)
        
        return src_out
    
    def decode(self, target, src_out, src_mask, target_mask):
        target = self.target_embed(target)
        target = self.target_pos(target)
        target_out = self.decoder(target, src_out, src_mask, target_mask)
        
        return target_out

In [13]:
# initializing transformer for language translation 

def init_transformer(src_vocab_size, target_vocab_size, 
                     src_seq_len, target_seq_len, d_model=512, 
                     num_layers=6, num_heads=8, dropout=0.1, d_ff=2048):
    
    src_embed = InputEmbeddings(src_vocab_size, d_model)
    target_embed = InputEmbeddings(target_vocab_size, d_model)
    
    src_pos = PositionalEncoding(src_seq_len, d_model, dropout)
    target_pos = PositionalEncoding(target_seq_len, d_model, dropout)
    
    encoder_stack = []
    decoder_stack = []
    
    for _ in range(num_layers):
        enc_self_attention = MultiHeadAttentionBlock(d_model, num_heads, dropout)
        enc_feed_forward_net = FeedForwardBLock(d_model, d_ff, dropout)
        
        encoder_block = EncoderBlock(d_model, enc_self_attention, enc_feed_forward_net, dropout)
        encoder_stack.append(encoder_block)
        
    
    for _ in range(num_layers):
        dec_self_attention = MultiHeadAttentionBlock(d_model, num_heads, dropout)
        dec_cross_attention = MultiHeadAttentionBlock(d_model, num_heads, dropout)
        dec_feed_forward_net = FeedForwardBLock(d_model, d_ff, dropout)
        
        decoder_block = DecoderBlock(d_model, dec_self_attention, dec_cross_attention, dec_feed_forward_net, dropout)
        decoder_stack.append(decoder_block)
        
    encoder = Encoder(nn.ModuleList(encoder_stack))
    decoder = Decoder(nn.ModuleList(decoder_stack))
    projection_layer = ProjectionLayer(d_model, target_vocab_size)
    
    transformer = Transformer(encoder, decoder, src_embed, target_embed, src_pos, target_pos, projection_layer)
    
    return transformer

In [14]:
def fetch_scentences(dataset, lang):
    return dataset['translation'][lang]

In [15]:
def load_tokenizer(config, dataset, lang):
    path_tokenizer = Path(config['tokenizer_file'].format(lang))
    
    if not Path.exists(path_tokenizer):
        tokenizer = Tokenizer(WordLevel(unk_token=['UNK']))
        tokenizer.pre_tokenizer = Whitespace()
        trainer = WordLevelTrainer(special_tokens=["[UNK]", "[PAD]", "[SOS]", "[EOS]"], min_frequency=2)
        tokenizer.train_from_iterator(fetch_scentences(dataset, lang), trainer)
        tokenizer.save(str(path_tokenizer))
    
    else:
        tokenizer = Tokenizer.from_file(str(path_tokenizer))
        
    return tokenizer

In [21]:
def load_dataset_(config):
    dataset_raw = load_dataset('opus_books', f'{config['lang_src']}-{config['lang_target']}', split='train')
    train_test_split = dataset_raw.train_test_split(test_size=0.2, seed=15)
    test_val_split = train_test_split['test'].train_test_split(test_size=0.5, seed=15)
    
    train_ds = train_test_split['train']
    val_ds = test_val_split['train']
    test_ds = test_val_split['test']
    
    tokenizer_src = load_tokenizer(config, train_ds, config['lang_src'])
    tokenizer_target = load_tokenizer(config, train_ds, config['lang_target'])
    
    return train_ds, val_ds, test_ds, tokenizer_src, tokenizer_target